# Build the fixed-core ESP-Gaussian cache

Reconstruct $\zeta=w\exp(-\rho^2/R_c^2)$ and compare Gaussian weighting with an equal-weight control. Both methods use the same physically fixed `FRAC=1` radius-of-maximum-tangential-velocity core. Run this notebook first on Katana.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import esp_pv_tools as ept
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}


In [ ]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
eddies, _ = tilt.load_tilt_tables(paths)
specs = ept.method_specs()

r = np.linspace(0, np.sqrt(0.5), 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
axes[0].plot(r, np.ones_like(r), color=".35", lw=3, label="Uniform")
axes[0].plot(r, np.exp(-r**2), color="tab:purple", lw=3, label="ESP Gaussian")
axes[0].axvline(np.sqrt(.5), color="k", ls="--", label="FRAC=1 boundary")
axes[0].set(xlabel=r"Elliptical radius $\rho/R_c$", ylabel="Cell weight", ylim=(0, 1.05),
            title="Weighting inside the fixed core")
axes[0].legend(frameon=False)
axes[1].bar(specs.method, specs.frac, color=[".35", "tab:purple"])
axes[1].axhline(1, color="k", ls="--")
axes[1].set(ylim=(0, 1.15), ylabel="FRAC", title="Identical physical sampling boundary")
plt.show()

In [ ]:
data = ept.build_cache(eddies, grid, specs)
destination = ept.cache_path()
destination.parent.mkdir(parents=True, exist_ok=True)
data.to_parquet(destination, index=False)

In [ ]:
counts = data.groupby("method").agg(snapshots=("Day","size"), eddies=("Eddy","nunique"),
    median_cells=("PV_footprint_n","median"), effective_cells=("PV_weight_effective_n","median")).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), constrained_layout=True)
sns.barplot(data=counts, x="method", y="snapshots", color=".35", ax=axes[0])
counts.plot(x="method", y=["median_cells","effective_cells"], kind="bar", ax=axes[1], color=[".65","tab:purple"])
for ax in axes: ax.tick_params(axis="x", rotation=20); ax.set_xlabel("")
axes[0].set_title("Snapshots retained")
axes[1].set_title("Same cells; different effective weighting")
plt.show()